# Step 7 — Export des qualifizierten Katalogs

Vereinigt den aktuellen Step-5-Output (Tier 1: aktuelle Nachtzug-Halte) mit den manuellen Step-6-Ergänzungen und schreibt eine CSV, die exakt dem Spaltenvertrag von `seed.py`s `_ONTD_SEED_CSV_COLUMNS` entspricht. Schlüssel ist die OSM-ID, sodass ein Stop, der auf beiden Wegen qualifiziert, nur einmal geschrieben wird und ein erneuter Step-5-Lauf direkt in den Katalog einfließt, ohne die manuelle Auswahl anzutasten.

Aus welcher Ebene ein Stop stammt, wird separat in `stop_seed_provenance.csv` geschrieben: Der Katalog selbst muss exakt dem Spaltenvertrag von `seed.py` entsprechen, daher kann die Provenienz nicht darin mitfahren. Ohne dieses Sidecar würde nichts auf der Platte zwischen einem Stop, den ein Nachtzug heute bedient, und einem manuell hinzugefügten unterscheiden.

**country** — stammt aus dem ONTD-Export via Step-4-Join, sofern vorhanden, da ONTD kuratierte nationale Daten sind; sonst der Wert der jeweiligen Quellebene. Steps 5 und 6 wenden dieselbe Präferenz bereits an, dies ist also ein Sicherheitsnetz und nicht die Stelle, an der die Korrektur passiert — Step 6 berichtet, was er geändert hat.

**stop_timezone** — aus dem Land als IANA-Name abgeleitet. Die alte Step-6-Datei trug stattdessen einen bloßen UTC-Offset, der keine Sommerzeit abbilden kann und für Irland falsch war (als +1 markiert); Schema und restlicher Katalog nutzen IANA-Namen.

**stop_charge_eur** stammt aus `charges/data/station_charges.csv`, generiert von den Kalibrierungs-Notebooks in `charges/`. Ein Stop, der dort fehlt, wird leer geschrieben, und `seed.py` setzt NULL, was über den Land-/Global-Default (aktuell 11.28 EUR) aufgelöst wird. Leer ist Absicht: eine Platzhalterzahl würde diesen Fallback überschreiben und die Frage "welche Stops brauchen noch echte Entgeltdaten?" unbeantwortbar machen, während NULL die Frage zu einer Einzeiler-Query macht.

## Imports

In [1]:
from __future__ import annotations

import csv
from pathlib import Path

from data_sources import DATA_DIR, ensure_local, local_input
from shapely.geometry import Point

import geopandas as gpd
import pandas as pd

## Pfade und Spaltenschema

In [38]:
OUTPUT_PATH = DATA_DIR / "stop_seed_catalog.csv"
PROVENANCE_PATH = DATA_DIR / "step6_manual_additions.csv"
COUNTRIES_SHP = "data/ne_10m_admin_0_countries.shp"

#für countrycode matching
NEAREST_MAX_DISTANCE_M = 2000

# Generiert von charges/02_station_charges.ipynb, das die registrierten
# Tarifdokumente einliest. Gitignored wie jedes andere Kalibrierungs-Artefakt
# im Projekt — die Notebooks sind die Wahrheit, dies ist ihr Output.
CHARGES_PATH = (
    Path.cwd() / "charges" / "data" / "station_charges.csv"
)

# seed.py::_ONTD_SEED_CSV_COLUMNS — im Gleichschritt halten.
SEED_COLUMNS = [
    "stop_id",
    "stop_name",
    "country_code",
    "stop_timezone",
    "stop_lat",
    "stop_lon",
    "stop_charge_eur",
]

In [16]:
#step6 laden und relevante Spalten wählen & zu Geodataframe umwandeln
step6_output = pd.read_csv(PROVENANCE_PATH, encoding="utf-8-sig")
stopdata = step6_output[[
    "stop_id",
    "stop_name",
    "stop_timezone",
    "stop_lat",
    "stop_lon"]]

geometry = [Point(xy) for xy in zip(stopdata["stop_lon"], stopdata["stop_lat"])]
stops_gdf = gpd.GeoDataFrame(stopdata.copy(), geometry=geometry, crs="EPSG:4326")

#shapefile mit ländergrenzen ladenb
countries = gpd.read_file(COUNTRIES_SHP).to_crs("EPSG:4326")

## Countrycode hinzufügen

In [46]:
def resolve_iso_a3(row):
    for field in ("ISO_A3", "ISO_A3_EH"):
        value = row.get(field)
        if value and value != "-99":
            return value
    return None

countries["country_code"] = countries.apply(resolve_iso_a3, axis=1)

MANUAL_ISO_FIXES = {
    "Norway": "NOR", "Kosovo": "XKX",
    "N. Cyprus": "CYP",
}
missing_iso = countries["country_code"].isna()
countries.loc[missing_iso, "country_code"] = countries.loc[missing_iso, "ADMIN"].map(MANUAL_ISO_FIXES)

countries_slim = countries[["country_code", "ADMIN", "geometry"]].rename(columns={"ADMIN": "country_name"})

In [24]:
missing_iso = countries["country_code"].isna()

In [26]:
#in welchen land leigen punkte + countrycode hinzufügen
joined = gpd.sjoin(stops_gdf, countries_slim, how="left", predicate="within")
joined = joined.drop(columns=["index_right"], errors="ignore")

In [48]:
#nähestes Land wählen für die die nciht gematcht werden konnten
unmatched_mask = joined["country_code"].isna()
unmatched = joined[unmatched_mask].copy()

if len(unmatched):
    unmatched_proj = unmatched.drop(columns=["country_code", "country_name"]).to_crs("EPSG:3857")
    countries_proj = countries_slim.to_crs("EPSG:3857")

    nearest = gpd.sjoin_nearest(
        unmatched_proj, countries_proj, how="left",
        max_distance=NEAREST_MAX_DISTANCE_M, distance_col="distance_m",
    )
    nearest = nearest.drop(columns=["index_right"], errors="ignore")

    for col in ["country_code", "country_name"]:
        joined.loc[unmatched_mask, col] = nearest[col].values

still_unmatched = joined[joined["country_code"].isna()]
if len(still_unmatched):
    ohne_land = joined[joined["country_code"].isna()]
    print(f"{len(ohne_land)} Stops ohne Land:")
    print(ohne_land[["stop_id", "stop_name", "stop_lat", "stop_lon"]])

stopdata_with_country = joined.drop(columns="geometry").reset_index(drop=True)

3 Stops ohne Land:
             stop_id        stop_name   stop_lat   stop_lon
583  osm:n5526331885  Bodø - Bådåddjo  67.286319  14.390816
585  osm:n5526332038           Narvik  68.441550  17.441113
591  osm:n5720557872        Stavanger  58.966577   5.732216


In [50]:
#manuell countrycode hinzufügen -> stopID von oben rauskopieren


MANUAL_COUNTRY_FIXES = {
    # "stop_id": "XX",
    "osm:n5526331885" : "NOR",
    "osm:n5526332038" : "NOR",
    "osm:n5720557872" : "NOR"
}
mask = stopdata_with_country["stop_id"].isin(MANUAL_COUNTRY_FIXES)
stopdata_with_country.loc[mask, "country_code"] = stopdata_with_country.loc[mask, "stop_id"].map(MANUAL_COUNTRY_FIXES)

In [52]:
stopdata_with_country.head()

,stop_id,stop_name,stop_timezone,stop_lat,stop_lon,country_code,country_name
0,osm:n25546152,Pécs,1,46.066366,18.225331,HUN,Hungary
1,osm:n13895194676,Terminali i Transportit Publik Tiranë,1,41.347259,19.777025,ALB,Albania
2,osm:n13895194677,Durrës,1,41.317741,19.456562,ALB,Albania
3,osm:n1613271652,Prishtinë,1,42.658934,21.151067,XKX,Kosovo
4,osm:n2107256271,Ferizaj,1,42.368744,21.153630,XKX,Kosovo


## Länder-Zeitzonen

Eine IANA-Zone pro Land. Aus dem Land abgeleitet statt aus einem UTC-Offset, damit die Sommerzeit von der Zonendatenbank gehandhabt wird, statt in den Daten eingefroren zu sein.

In [53]:
COUNTRY_TIMEZONES = {
    "ALB": "Europe/Tirane",
    "AUT": "Europe/Vienna",
    "BIH": "Europe/Sarajevo",
    "BEL": "Europe/Brussels",
    "BGR": "Europe/Sofia",
    "BLR": "Europe/Minsk",
    "CHE": "Europe/Zurich",
    "CZE": "Europe/Prague",
    "DEU": "Europe/Berlin",
    "DNK": "Europe/Copenhagen",
    "EST": "Europe/Tallinn",
    "ESP": "Europe/Madrid",
    "FIN": "Europe/Helsinki",
    "FRA": "Europe/Paris",
    "GBR": "Europe/London",
    "GRC": "Europe/Athens",
    "HRV": "Europe/Zagreb",
    "HUN": "Europe/Budapest",
    "IRL": "Europe/Dublin",
    "ITA": "Europe/Rome",
    "LTU": "Europe/Vilnius",
    "LUX": "Europe/Luxembourg",
    "LVA": "Europe/Riga",
    "MDA": "Europe/Chisinau",
    "MNE": "Europe/Podgorica",
    "MKD": "Europe/Skopje",
    "NLD": "Europe/Amsterdam",
    "NOR": "Europe/Oslo",
    "POL": "Europe/Warsaw",
    "PRT": "Europe/Lisbon",
    "ROU": "Europe/Bucharest",
    "SRB": "Europe/Belgrade",
    "RUS": "Europe/Moscow",
    "SWE": "Europe/Stockholm",
    "SVN": "Europe/Ljubljana",
    "SVK": "Europe/Bratislava",
    "TUR": "Europe/Istanbul",
    "UKR": "Europe/Kyiv",
    "XKX": "Europe/Belgrade",
}

stopdata_with_country["stop_timezone"] = stopdata_with_country["country_code"].map(COUNTRY_TIMEZONES)

## Ausgeschlossene Stop-IDs

OSM-IDs, deren Step-4-Match Ausschuss ist — die OSM-Station ist ein unbenanntes oder einbuchstabiges Objekt Hunderte Kilometer vom ONTD-Stop entfernt, dem sie zugeordnet wurde. Ausgeschlossen statt mit falschem Standort geseedet.

In [ ]:
EXCLUDED_STOP_IDS = {
    "osm:n4896717721",  # ONTD Dağkadı Hızlı Tren İstasyonu -> OSM "tren", 2743 km
    "osm:n8515238217",  # ONTD Tekučica -> OSM "A", 355 km
    "osm:n9553124517",  # ONTD Közép-Garadna -> OSM "Arad", 222 km
}

## ... weiter noch nicht bearbeitet (Johanna) ...

## Funktion: Stationsentgelte laden

In [ ]:
def load_station_charges() -> dict[str, float]:
    """stop_id -> Entgelt in EUR, aus der versionierten station_charges.csv.

    Fehlende Datei ist kein Fehler: der Katalog wird dann komplett über den
    Default geseedet, was der ehrliche Zustand ist, bis echte Tarife erhoben
    wurden.
    """
    if not CHARGES_PATH.is_file():
        print(
            f"  {CHARGES_PATH.name} nicht gefunden — jedes stop_charge_eur bleibt "
            "leer, sodass jeder Stop über den globalen Default aufgelöst wird. "
            "charges/01_source_extraction.ipynb dann "
            "02_station_charges.ipynb ausführen, um sie zu generieren."
        )
        return {}

    charges: dict[str, float] = {}
    illustrative = 0
    with open(CHARGES_PATH, encoding="utf-8-sig", newline="") as fh:
        for line, row in enumerate(csv.DictReader(fh), start=2):
            stop_id = (row.get("stop_id") or "").strip()
            value = parse_float(row.get("stop_charge_eur"))
            if not stop_id or value is None:
                continue
            if value < 0:
                raise ValueError(f"{CHARGES_PATH.name} Zeile {line}: negatives Entgelt")
            if stop_id in charges:
                raise ValueError(
                    f"{CHARGES_PATH.name} Zeile {line}: doppelte {stop_id}"
                )
            charges[stop_id] = value
            # Die Entgelt-Notebooks registrieren nicht belegte Übernahmewerte
            # unter dieser ID; gezählt, damit ein Lauf offen ausweist, wie viel
            # noch Platzhalter statt gemessen ist.
            if (row.get("source_ref") or "").strip() == "ILLUSTRATIVE-CURATED":
                illustrative += 1

    print(f"  {len(charges)} Stationsentgelte ({illustrative} noch illustrativ).")
    return charges

## Funktion: Float parsen

In [ ]:
def parse_float(value):
    if value is None:
        return None
    text = str(value).strip().replace(",", ".")
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None

## Funktion: ONTD-Länder laden

In [ ]:
def load_ontd_countries() -> dict[str, str]:
    """OSM-Stop-ID -> ONTD-Ländercode, aus dem Step-4-Join."""
    path = ensure_local("step4_MatchingONTDtoOSM.csv")
    countries = {}
    with open(path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            osm_id = (row.get("osm_stop_id") or "").strip()
            country = (row.get("ontd_country") or "").strip().upper()
            if osm_id and country:
                countries.setdefault(osm_id, country)
    return countries

## Funktion: Kandidaten iterieren (Step 5, dann Step 6)

In [ ]:
def iter_candidates():
    """(source, stop_id, stop_name, source_country, lat, lon, reason) aus
    Step 5, dann Step 6. Step 5 zuerst, damit seine ONTD-gestützte Zeile
    gewinnt, wenn ein Stop auf beiden Wegen qualifiziert."""
    step5_path = local_input(
        "step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"
    )
    with open(step5_path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            yield (
                "step5",
                (row.get("osm_stop_id") or "").strip(),
                (row.get("osm_stop_name") or row.get("ontd_name") or "").strip(),
                (row.get("ontd_country") or "").strip().upper(),
                parse_float(row.get("osm_lat")),
                parse_float(row.get("osm_lon")),
                f"night_train_stop:{(row.get('schedule_name') or '').strip()}",
            )
    step6_path = local_input(
        "step6_manual_additions.csv", "step6_manual_additions.ipynb"
    )
    with open(step6_path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            yield (
                "step6_manual",
                (row.get("stop_id") or "").strip(),
                (row.get("stop_name") or "").strip(),
                (row.get("country") or "").strip().upper(),
                parse_float(row.get("stop_lat")),
                parse_float(row.get("stop_lon")),
                (row.get("reason") or "").strip(),
            )

## Daten laden: ONTD-Länder und Entgelte

In [ ]:
ontd_countries = load_ontd_countries()
charges = load_station_charges()

## Kandidaten verarbeiten: Katalog- und Provenienz-Zeilen bauen

Dedupliziert per OSM-ID, filtert ausgeschlossene Stops, löst Land/Zeitzone auf und sammelt Stops ohne auflösbare Zeitzone für den Abbruch am Ende.

In [ ]:
rows, unknown_tz = [], []
skipped: dict[str, str] = {}
seen_ids = set()
per_source = {"step5": 0, "step6_manual": 0}
provenance = []

for (
    source,
    stop_id,
    stop_name,
    source_country,
    lat,
    lon,
    reason,
) in iter_candidates():
    if not stop_id or stop_id in seen_ids:
        continue
    if stop_id in EXCLUDED_STOP_IDS:
        skipped.setdefault(stop_id, stop_name)
        continue
    seen_ids.add(stop_id)
    per_source[source] += 1

    country = ontd_countries.get(stop_id, source_country)

    timezone = COUNTRY_TIMEZONES.get(country)
    if timezone is None:
        # Kein Land bedeutet keine Zeitzone, und ein ohne sie geschriebener
        # Stop wäre falsch statt nur unvollständig — daher hier gesammelt und
        # unten ausgelöst, damit er nie unbemerkt aus dem Katalog verschwindet.
        unknown_tz.append((stop_id, stop_name, country))
        continue

    if lat is None or lon is None:
        skipped[stop_id] = f"{stop_name} (fehlende Koordinaten)"
        continue

    provenance.append(
        {
            "stop_id": stop_id,
            "stop_name": stop_name,
            "source": source,
            "reason": reason,
        }
    )
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": stop_name,
            "country_code": country,
            "stop_timezone": timezone,
            "stop_lat": f"{lat:.7f}",
            "stop_lon": f"{lon:.7f}",
            "stop_charge_eur": (
                "" if stop_id not in charges else f"{charges[stop_id]:.2f}"
            ),
        }
    )

## Katalog und Provenienz schreiben

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=SEED_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

with open(PROVENANCE_PATH, "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(
        fh, fieldnames=["stop_id", "stop_name", "source", "reason"]
    )
    writer.writeheader()
    writer.writerows(provenance)

print(
    f"{len(rows)} Stops nach {OUTPUT_PATH.name} geschrieben "
    f"({per_source['step5']} aus Step 5, "
    f"{per_source['step6_manual']} manuelle Step-6-Ergänzungen)"
)

## Zusammenfassung und Warnungen

Bricht am Ende mit `SystemExit` ab, falls Stops ohne auflösbare Zeitzone übrig blieben — damit ein solcher Stop nie unbemerkt aus dem Katalog verschwindet.

In [ ]:
unexplained = sum(
    1 for p in provenance if p["source"] == "step6_manual" and not p["reason"]
)
if unexplained:
    print(
        f"  {unexplained} manuelle Ergänzungen tragen keinen Grund "
        f"— siehe step6_manual_additions.ipynb"
    )
if skipped:
    print(f"übersprungen {len(skipped)}: {skipped}")
unknown_charges = sorted(set(charges) - seen_ids)
if unknown_charges:
    print(
        f"  WARNUNG: {len(unknown_charges)} Entgelt-Zeile(n) nennen einen Stop, "
        f"der nicht im Katalog ist — veraltete IDs in {CHARGES_PATH.name}: "
        f"{unknown_charges[:5]}"
    )

if unknown_tz:
    countries = sorted({country for _, _, country in unknown_tz if country})
    blank = [(i, n) for i, n, country in unknown_tz if not country]
    raise SystemExit(
        f"{len(unknown_tz)} Stop(s) mangels Zeitzone verworfen.\n"
        + (
            f"  keine Zuordnung für {countries} — zu COUNTRY_TIMEZONES hinzufügen\n"
            if countries
            else ""
        )
        + (
            f"  gar kein Land: {blank} — upstream in Step 6 beheben\n"
            if blank
            else ""
        )
    )